In [2]:
import pandas as pd
import numpy as np
import os
import gc
import transformers
import torch
import logging
import warnings
import json

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report
from pathlib import Path
from tqdm.notebook import tqdm

logging.getLogger("transformers").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")
tqdm.pandas()

In [3]:
CATEGORIES = ["Мир", "Россия", "Экономика", "Наука и техника", "Спорт", "Культура"]
CATEGORIES_STR = ", ".join(CATEGORIES)

N_SHOTS    = 2    # примеров на класс
EXAMPLE_LEN = 300  # символов в каждом примере

DATA_PATH   = Path("news_data/cleaned_news_for_model.parquet")
SAMPLE_PATH = Path("news_data/llm_sample_1200.parquet")
REPORT_PATH = Path("reports/llm_fewshot_results.csv")
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)

In [4]:
# Загружаем ту же выборку что использовалась в zero-shot
df_sample = pd.read_parquet(SAMPLE_PATH)
print(f"Тестовая выборка: {len(df_sample)} строк")

# Для примеров берём всё что НЕ попало в выборку
df = pd.read_parquet(DATA_PATH)
df_model = df[["title", "text", "category_raw"]].dropna().copy()
df_model["llm_text"] = df_model["text"].astype(str)
df_model = df_model[df_model["llm_text"].str.len() > 0]

df_pool = df_model[~df_model.index.isin(df_sample.index)]
print(f"Пул для примеров: {len(df_pool)} строк")

Тестовая выборка: 1200 строк
Пул для примеров: 145419 строк


In [5]:
few_shot_examples = []
for cat in CATEGORIES:
    cat_examples = (
        df_pool[df_pool["category_raw"] == cat]
        .sample(n=N_SHOTS, random_state=42)
    )
    for _, row in cat_examples.iterrows():
        few_shot_examples.append({
            "category": cat,
            "text": row["llm_text"][:EXAMPLE_LEN],
        })

# Проверяем что примеры не пересекаются с тестовой выборкой по индексу
example_idx = {ex["text"] for ex in few_shot_examples}
leaks = df_sample["llm_text"].apply(lambda t: t[:EXAMPLE_LEN] in example_idx).sum()
assert leaks == 0, f"Утечка! {leaks} примеров из промпта есть в выборке"

print(f"Few-shot примеров: {len(few_shot_examples)} ({N_SHOTS} на класс × {len(CATEGORIES)} классов)")
print(f"Каждый пример — первые {EXAMPLE_LEN} символов")

# Показываем что получилось
for ex in few_shot_examples:
    print(f"\n[{ex['category']}] {ex['text'][:80]}...")

Few-shot примеров: 12 (2 на класс × 6 классов)
Каждый пример — первые 300 символов

[Мир] Politico: ЕС разрабатывает план по частичному членству Украины в 2027 году
Семен...

[Мир] Sky News: Лидеры ЕС могут поехать в США, чтобы повлиять на Трампа по Украине
Вик...

[Россия] Сенатор Косачев: Превращение БРИКС или ШОС в военный блок бесперспективно
Идеи п...

[Россия] Врачи НИИ Склифосовского провели уникальную трансплантацию кисти от донора
Росси...

[Экономика] Тайфун образовался к югу от Японии, он может затронуть Курильские острова
Тайфун...

[Экономика] Депутат Госдумы Нилов: Цены на цветы следует ограничить законодательно
Фото: Ser...

[Наука и техника] Daily Mail: Распятие Иисуса Христа могло произойти 3 апреля 33 года нашей эры
Ек...

[Наука и техника] eBioMedicine: У пациентов с длительным COVID повышается уровень тау-белка
Екатер...

[Спорт] Вратарь сборной России по водному поло Федотов: Были готовы играть с Украиной
Фо...

[Спорт] Лыжник Коростелев выполнил олимпийский нормат

In [6]:
SYSTEM_PROMPT_FS = (
    f"Ты классификатор новостных статей.\n"
    f"Определи категорию текста и ответь ТОЛЬКО одним из вариантов: {CATEGORIES_STR}.\n"
    f"Никаких пояснений — только одно слово или фраза из списка."
)

def build_few_shot_messages(text, cache_file="news_data/prompts/few_shots_2_per_class_300_symb.json"):
    """
    Собирает messages: system + примеры как диалог + целевой текст.
    
    Кеш хранит только базовую часть (system + few-shot примеры).
    Целевой text всегда добавляется в конец динамически.
    """
    if cache_file is not None:
        cache_path = Path(cache_file)

        if cache_path.exists():
            with cache_path.open("r", encoding="utf-8") as f:
                base_messages = json.load(f)
            return base_messages + [{"role": "user", "content": text}]

    # Строим базовую часть без целевого текста
    base_messages = [{"role": "system", "content": SYSTEM_PROMPT_FS}]
    for ex in few_shot_examples:
        base_messages.append({"role": "user",      "content": ex["text"]})
        base_messages.append({"role": "assistant", "content": ex["category"]})

    # Сохраняем только базовую часть
    if cache_file is not None:
        cache_path = Path(cache_file)
        cache_path.parent.mkdir(parents=True, exist_ok=True)
        with cache_path.open("w", encoding="utf-8") as f:
            json.dump(base_messages, f, ensure_ascii=False, indent=2)

    return base_messages + [{"role": "user", "content": text}]

def classify_few_shot(text, pipeline, max_new_tokens=20):
    messages = build_few_shot_messages(text)
    output = pipeline(messages, max_new_tokens=max_new_tokens, do_sample=False)
    response = output[0]["generated_text"][-1]["content"].strip()

    for cat in CATEGORIES:
        if cat.lower() in response.lower():
            return cat

    print(f"[Неизвестно] Ответ модели: '{response}'")
    return "Неизвестно"

def free_memory(obj):
    del obj
    gc.collect()
    torch.mps.empty_cache()
    print("Память освобождена")

def compute_metrics(y_true, y_pred, model_name):
    mask = y_pred != "Неизвестно"
    accuracy    = accuracy_score(y_true[mask], y_pred[mask])
    macro_f1    = f1_score(y_true[mask], y_pred[mask], average="macro",    zero_division=0)
    weighted_f1 = f1_score(y_true[mask], y_pred[mask], average="weighted", zero_division=0)
    unknown_rate = (~mask).mean()

    print(f"\n=== {model_name} ===")
    print(f"Accuracy:    {accuracy:.4f}")
    print(f"Macro F1:    {macro_f1:.4f}")
    print(f"Weighted F1: {weighted_f1:.4f}")
    print(f"Неизвестно:  {unknown_rate:.1%} ответов не распознано")
    print(classification_report(y_true[mask], y_pred[mask], zero_division=0))

    return accuracy, macro_f1, weighted_f1, unknown_rate

In [7]:
# Посмотреть как выглядит промпт перед запуском
test_messages = build_few_shot_messages("Тестовый текст новости")
for msg in test_messages:
    role = msg["role"].upper()
    preview = msg["content"][:100].replace("\n", " ")
    print(f"[{role}] {preview}...")
    print()

[SYSTEM] Ты классификатор новостных статей. Определи категорию текста и ответь ТОЛЬКО одним из вариантов: Мир...

[USER] Politico: ЕС разрабатывает план по частичному членству Украины в 2027 году Семен Александров (старши...

[ASSISTANT] Мир...

[USER] Sky News: Лидеры ЕС могут поехать в США, чтобы повлиять на Трампа по Украине Виктория Кондратьева (Р...

[ASSISTANT] Мир...

[USER] Сенатор Косачев: Превращение БРИКС или ШОС в военный блок бесперспективно Идеи превратить БРИКС или ...

[ASSISTANT] Россия...

[USER] Врачи НИИ Склифосовского провели уникальную трансплантацию кисти от донора Российские врачи НИИ скор...

[ASSISTANT] Россия...

[USER] Тайфун образовался к югу от Японии, он может затронуть Курильские острова Тайфун «Нари», пятый в это...

[ASSISTANT] Экономика...

[USER] Депутат Госдумы Нилов: Цены на цветы следует ограничить законодательно Фото: Sergey Elagin / Busines...

[ASSISTANT] Экономика...

[USER] Daily Mail: Распятие Иисуса Христа могло произойти 3 апреля 33 года н

In [8]:
pipeline_qwen = transformers.pipeline(
    "text-generation",
    model="Qwen/Qwen2.5-7B-Instruct",
    dtype="auto",
    device_map="auto",
)
print("Qwen загружен")

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Qwen загружен


In [9]:
print(f"Few-shot классификация {len(df_sample)} текстов через Qwen...")
df_sample["pred_qwen_fs"] = df_sample["llm_text"].progress_apply(
    lambda text: classify_few_shot(text, pipeline_qwen)
)
df_sample["pred_qwen_fs"].value_counts()

Few-shot классификация 1200 текстов через Qwen...


  0%|          | 0/1200 [00:00<?, ?it/s]

[Неизвестно] Ответ модели: 'Среда обитания'
[Неизвестно] Ответ модели: 'Силовые структуры'
[Неизвестно] Ответ модели: 'Среда обитания'


pred_qwen_fs
Мир                370
Россия             294
Экономика          285
Культура            98
Спорт               84
Наука и техника     66
Неизвестно           3
Name: count, dtype: int64

In [10]:
free_memory(pipeline_qwen)

Память освобождена


In [11]:
acc_q, mf1_q, wf1_q, unk_q = compute_metrics(
    df_sample["category_raw"], df_sample["pred_qwen_fs"], "Qwen2.5-7B-Instruct few-shot"
)


=== Qwen2.5-7B-Instruct few-shot ===
Accuracy:    0.8312
Macro F1:    0.8251
Weighted F1: 0.8317
Неизвестно:  0.2% ответов не распознано
                 precision    recall  f1-score   support

       Культура       0.58      0.90      0.71        63
            Мир       0.89      0.89      0.89       371
Наука и техника       0.91      0.72      0.81        83
         Россия       0.83      0.72      0.77       338
          Спорт       0.93      0.96      0.95        81
      Экономика       0.80      0.87      0.83       261

       accuracy                           0.83      1197
      macro avg       0.82      0.84      0.83      1197
   weighted avg       0.84      0.83      0.83      1197



In [12]:
results = pd.DataFrame([

    {
        "experiment": "14_qwen2.5_7b_fewshot",
        "model": "Qwen/Qwen2.5-7B-Instruct",
        "input": "text", "classifier": f"few-shot LLM ({N_SHOTS}/class)",
        "sample_size": len(df_sample),
        "accuracy": acc_q, "macro_f1": mf1_q,
        "weighted_f1": wf1_q, "unknown_rate": unk_q,
    },

])

results.to_csv(
    REPORT_PATH,
    mode="a",
    header=not REPORT_PATH.exists(),
    index=False,
)
results

,experiment,model,input,classifier,sample_size,accuracy,macro_f1,weighted_f1,unknown_rate
0,14_qwen2.5_7b_fewshot,Qwen/Qwen2.5-7B-Instruct,text,few-shot LLM (2/class),1200,0.831245,0.82509,0.831654,0.0025


In [17]:
model_col = "pred_qwen_fs"  

errors = df_sample[df_sample["category_raw"] != df_sample[model_col]][[
    "llm_text", "category_raw", model_col
]].rename(columns={"llm_text": "text", "category_raw": "true", model_col: "pred"})

print(f"Ошибок: {len(errors)} из {len(df_sample)}")
with pd.option_context("display.max_colwidth", 300):
    display(errors.head(20))

Ошибок: 205 из 1200


,text,true,pred
107059,"Фитнес-тренер упомянула еду после 18:00 и вред углеводов среди мифов о похудении\nФитнес-тренер Ксения Гуфранова назвала топ мифов о похудении. Об этом сообщают «Известия» . Самое распространенное заблуждение, по мнению специалиста, связано с ограничениями по времени. «Многие уверены, что ужины ...",Спорт,Экономика
26262,"На Западном берегу Иордана избили палестинского режиссера Хамдана Баллала\nФото: Monika Skolimowska / Globallookpress\nФото: Monika Skolimowska / Globallookpress\nНа Западном берегу реки Иордан израильские поселенцы избили палестинского режиссера Хамдана Баллала, снявшего документальный фильм «Н...",Культура,Мир
64495,РИА: Конфликт Таиланда и Камбоджи ударит по поставкам кормов для кошек и собак\nКонфликт Таиланда и Камбоджи может ударить по поставкам кормов для домашних животных. Угрозу для кошек и собак увидели РИА Новости после анализа данных платформы ООН Comtrade. В 2024 году Таиланд занимал второе место...,Экономика,Мир
140930,"Доцент Балынин: Шестидневных рабочих недель в России в 2026 году не будет\nВ 2026 году для жителей России, работающих по стандартному пятидневному графику, введение шестидневной рабочей недели не планируется. Об этом РИА Новости заявил доцент кафедры общественных финансов Финансового университет...",Россия,Экономика
50050,"Средства от продаж в День мороженого в ГУМе переведут фонду «Наука – детям»\nВ московском ГУМе, прошел традиционный День мороженого. Праздник мороженого прошел под названием «""Артек"" на все 100!». Дело в том, что легендарный международный детский центр отмечает в этом году юбилей. Этому событию ...",Россия,Культура
88813,"При пожаре в Сосновоборске погиб один человек\nВ результате пожара в пятиэтажном доме в Сосновоборске Красноярского края погиб один человек. Об этом сообщает Telegram -канал Baza. Как стало известно, пожар начался в одной из квартир, а затем огонь перебросился на крышу. «Погиб один человек, еще ...",Россия,Экономика
36585,"Си Цзиньпин заявил, что технологии ИИ несут беспрецедентные риски\nВарвара Кошечкина (редактор отдела оперативной информации)\nВарвара Кошечкина (редактор отдела оперативной информации)\nТехнологии искусственного интеллекта (ИИ) предоставляют колоссальные возможности, однако при этом несут беспр...",Наука и техника,Экономика
56468,Shot: Четыре человека не выжили при падении самолета Як-18 в Подмосковье\nМарк Успенский (Редактор отдела «Путешествия»)\nМарк Успенский (Редактор отдела «Путешествия»)\nЛегкомоторный самолет Як-18 упал в Подмосковье . Об этом сообщил Telegram-канал Shot . Авария произошла недалеко от деревни Па...,Россия,Культура
75911,Машков: За два года «Театральный бульвар» стремительно завоевал любовь зрителей\nИдея «Театрального бульвара» перспективна для всех участников фестиваля. Об этом рассказалпредседатель Союза театральных деятелей РФ Владимир Машков . «Театральный бульвар» очень успешный проект Москвы . За эти два ...,Россия,Культура
69832,"Подольская заявила о праве россиян взять больничный и при отсутствии болезни\nЭксперт РАНХиГС Татьяна Подольская заявила, что россияне могут оформить больничный лист, даже если не болеют. Об этом она высказалась в беседе с РИА Новости . Специалист заявила о праве россиян взять больничный и при о...",Россия,Экономика


In [18]:
unknown = df_sample[df_sample[model_col] == "Неизвестно"][["llm_text", "category_raw", model_col]]\
    .rename(columns={"llm_text": "text", "category_raw": "true", model_col: "pred"})

print(f"Неизвестно: {len(unknown)}")
with pd.option_context("display.max_colwidth", 800):
    display(unknown)

Неизвестно: 3


,text,true,pred
116620,"Синоптик Вильфанд: К выходным в Москве и области похолодает до минус 25 градусов\nНина Ташевская (Редактор отдела «Среда обитания»)\nНина Ташевская (Редактор отдела «Среда обитания»)\nК предстоящим выходным, 17 и 18 января, в Москве и области может похолодать до минус 25 градусов. О резком снижении температуры предупредил жителей столичного региона научный руководитель Гидрометцентра России Роман Вильфанд , пишет ТАСС. При этом до пятницы включительно, по прогнозам синоптика, ожидается хорошая зимняя погода — температура воздуха будет на 1-2 градуса ниже климатической нормы. Ночью столбики термометров покажут минус 7-11 градусов, а днем — минус 6-8 градусов. «А вот дальше, начиная с выходных, в отдельных районах, скажем, на юго-востоке Московской области, не исключено понижение темпера...",Экономика,Неизвестно
93561,"В Москве 12-летний мальчик на машине врезался в дерево\nАртем Соколов (Редактор отдела «Силовые структуры»)\nАртем Соколов (Редактор отдела «Силовые структуры»)\nФото: Telegram-канал Прокуратуры г. Москвы\nФото: Telegram-канал Прокуратуры г. Москвы\nВ Москве 12-летний мальчик на машине врезался в дерево. Об этом «Ленте.ру» сообщили в столичной прокуратуре. По данным ведомства, в Балакиревском переулке мальчик, управляя автомобилем «Лада Ларгус», не справился с управлением и совершил столкновение с деревом. В результате происшествия пострадали он и его 15-летняя подруга. Оба получили различные травмы и были госпитализированы. После дорожно-транспортного происшествия (ДТП) подросток рассказал, что перепутал педали. По факту ДТП проводится проверка, направленная на установление всех обсто...",Россия,Неизвестно
63408,"Дептранс: В Москве из-за ливней перекрыли движение по нескольким участкам дорог\nНина Ташевская (Редактор отдела «Среда обитания»)\nНина Ташевская (Редактор отдела «Среда обитания»)\nВ Москве из-за ливней перекрыли движение по нескольким участкам автодорог. Об этом сообщает столичный Дептранс в своем Telegram -канале. Так, из-за погодных условий машины временно не могут проехать по Волоколамскому шоссе в районе дома 142, по проспекту Мира в районе дома 133 (под мостом в районе реки Яуза), а также по улице Тихая. «Просим строить маршрут заранее. За ситуацией на дороге следит Ситуационный центр ЦОДД», — добавили в ведомстве. Ранее на Москву обрушились сильные ливни. За ночь и утро понедельника, 21 июля, в столице выпала месячная норма осадков. Синоптики уточняют, что непогода продлится д...",Экономика,Неизвестно
